# 120 Years of Olympic History

Grasso Gianni & Ismael Trentin

## Dataset

This is a historical dataset on the modern Olympic Games, including all the Games from Athens 1896 to Rio 2016. The data was scraped from www.sports-reference.com in May 2018. More info can be found [here](https://www.kaggle.com/datasets/heesoo37/120-years-of-olympic-history-athletes-and-results).


## Code

In [21]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# pio.renderers.default = "notebook"
from plotly.subplots import make_subplots

In [22]:
df = pd.read_csv('data/athlete_events.csv')

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  str    
 2   Sex     271116 non-null  str    
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  str    
 7   NOC     271116 non-null  str    
 8   Games   271116 non-null  str    
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  str    
 11  City    271116 non-null  str    
 12  Sport   271116 non-null  str    
 13  Event   271116 non-null  str    
 14  Medal   39783 non-null   str    
dtypes: float64(3), int64(2), str(10)
memory usage: 31.0 MB


Our dataset records a participation in an olympic game, meaning an athlete could appear multiple times.

In [24]:
df[df["Name"].str.contains("Usain St. Leo Bolt", case=False, na=False)]

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
24876,13029,Usain St. Leo Bolt,M,17.0,196.0,95.0,Jamaica,JAM,2004 Summer,2004,Summer,Athina,Athletics,Athletics Men's 200 metres,NaN
24877,13029,Usain St. Leo Bolt,M,21.0,196.0,95.0,Jamaica,JAM,2008 Summer,2008,Summer,Beijing,Athletics,Athletics Men's 100 metres,Gold
24878,13029,Usain St. Leo Bolt,M,21.0,196.0,95.0,Jamaica,JAM,2008 Summer,2008,Summer,Beijing,Athletics,Athletics Men's 200 metres,Gold
24879,13029,Usain St. Leo Bolt,M,21.0,196.0,95.0,Jamaica,JAM,2008 Summer,2008,Summer,Beijing,Athletics,Athletics Men's 4 x 100 metres Relay,NaN
24880,13029,Usain St. Leo Bolt,M,25.0,196.0,95.0,Jamaica,JAM,2012 Summer,2012,Summer,London,Athletics,Athletics Men's 100 metres,Gold
24881,13029,Usain St. Leo Bolt,M,25.0,196.0,95.0,Jamaica,JAM,2012 Summer,2012,Summer,London,Athletics,Athletics Men's 200 metres,Gold
24882,13029,Usain St. Leo Bolt,M,25.0,196.0,95.0,Jamaica,JAM,2012 Summer,2012,Summer,London,Athletics,Athletics Men's 4 x 100 metres Relay,Gold
24883,13029,Usain St. Leo Bolt,M,29.0,196.0,95.0,Jamaica,JAM,2016 Summer,2016,Summer,Rio de Janeiro,Athletics,Athletics Men's 100 metres,Gold
24884,13029,Usain St. Leo Bolt,M,29.0,196.0,95.0,Jamaica,JAM,2016 Summer,2016,Summer,Rio de Janeiro,Athletics,Athletics Men's 200 metres,Gold
24885,13029,Usain St. Leo Bolt,M,29.0,196.0,95.0,Jamaica,JAM,2016 Summer,2016,Summer,Rio de Janeiro,Athletics,Athletics Men's 4 x 100 metres Relay,Gold


We decided to merge the different historically separated nations to have a better present picture. 

In [25]:
# NOC 'harmonization'
# we unite geopolitical differences to have a global present view of the data
noc_mapping = {
    # Soviet Union
    "URS": "RUS",
    "EUN": "RUS", 
    
    # Germany
    "GDR": "GER",
    "FRG": "GER",
    
    # Yugoslavia
    "YUG": "SRB",
    
    # Czechoslovakia
    "TCH": "CZE"
}

df["NOC"] = df["NOC"].replace(noc_mapping)

In [26]:
# drop na values
medals = df[df["Medal"].notna()]

# since we have a record for each participation,
# team games register the same medal multiple times,
# boosting the actual count for that medal.
medals = medals.drop_duplicates(
    subset=["NOC", "Games", "Event", "Medal"]
)

# for each NOC count each Medal type and flatmap that using reset_index
# (long format)
top_noc = (medals.groupby(["NOC", "Medal"])
                 .size()
                 .reset_index(name="count"))
# count NOC occs (auto desc order) and take first 20 (just NOCs)
top20 = medals["NOC"].value_counts().head(20).index
top_noc = top_noc[top_noc["NOC"].isin(top20)]

fig1 = px.bar(
    top_noc,
    x="NOC",
    y="count",
    color="Medal",
    color_discrete_map={"Gold": "#FFD700", "Silver": "#C0C0C0", "Bronze": "#CD7F32"},
    category_orders={
      "Medal": ["Gold", "Silver", "Bronze"],
      "NOC": top20,
    },
    title="Top 20 Nations by Total Medals",
    labels={"count": "Medals [#]", "NOC": "Nation [NOC]"},
    template="plotly_white"
)
fig1.show()

In [27]:
medals_df = df[df["Medal"].notna()].copy()

# since we have a record for each participation,
# team games register the same medal multiple times,
# boosting the actual count for that medal.
medals_df = medals_df.drop_duplicates(
    subset=["NOC", "Games", "Event", "Medal"]
)

# keep NOC - Gold - Silver - Bronze headers and fill with group size
# (wide format)
medal_counts = (
    medals_df
    .groupby(["NOC", "Medal"])
    .size()
    .unstack(fill_value=0)
)

# ensure all medal columns exist
for medal in ["Gold", "Silver", "Bronze"]:
    if medal not in medal_counts.columns:
        medal_counts[medal] = 0

# compute total for each NOC
medal_counts["Total"] = medal_counts["Gold"] + medal_counts["Silver"] + medal_counts["Bronze"]

# top 20 by total
medal_counts = (medal_counts
                .sort_values("Total", ascending=False)
                .head(20))

# calc %
for medal in ["Gold", "Silver", "Bronze"]:
    medal_counts[medal+"_prc"] = medal_counts[medal] / medal_counts["Total"] * 100

# order those 20 by Gold %
medal_counts = (medal_counts
                .sort_values(by="Gold_prc", ascending=False))

# chart
fig = go.Figure()

# gold bars
fig.add_trace(
  go.Bar(
    x=medal_counts.index,
    y=medal_counts["Gold_prc"],
    name="Gold",
    marker_color="gold",
    hovertemplate=
        "<b>%{x}</b><br>" +
        "Gold: %{y:.1f}%<extra></extra>"
))

# silver bars
fig.add_trace(
    go.Bar(
      x=medal_counts.index,
      y=medal_counts["Silver_prc"],
      name="Silver",
      marker_color="silver",
      hovertemplate=
          "<b>%{x}</b><br>" +
          "Silver: %{y:.1f}%<extra></extra>"
))

# bronze bars
fig.add_trace(
    go.Bar(
      x=medal_counts.index,
      y=medal_counts["Bronze_prc"],
      name="Bronze",
      marker_color="#CD7F32",
      hovertemplate=
          "<b>%{x}</b><br>" +
          "Bronze: %{y:.1f}%<extra></extra>"
))

# stacked % bars
fig.update_layout(
    title="Top 20 Nations by Total Medals, Ordered by Gold %",
    barmode="stack",
    template="plotly_white",
    xaxis_title="Nation [NOC]",
    yaxis_title="Total Medals [%]",
    legend_title="Medal",
    height=650,
    width=1100
)
for noc, total in zip(medal_counts.index, medal_counts["Total"]):
    fig.add_annotation(
        x=noc,
        y=100,
        yshift=12,
        text=str(int(total)),
        opacity=0.7,
        showarrow=False
    )

fig.update_yaxes(
    range=[0, 100],
    ticksuffix="%"
)

fig.show()

In [28]:
# Year - Sex - total count
part = (df.groupby(["Year", "Sex"])
          .size()
          .reset_index(name="count"))
# Year(idx) - total count
year_total = (part.groupby("Year")["count"]
              .sum()
              .reset_index(name="total")
              .set_index("Year"))

# subplots config
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.45, 0.55]
)
color_sex = {"M": "#4C72B0", "F": "#DD52CD"}
label_sex = {"M": "Men", "F": "Women"}

# females %
# all females count per year
females = part[part["Sex"] == "F"].set_index("Year")
# all females % per year
         # calc %
pct_f = [females.loc[yr, "count"] / year_total.loc[yr, "total"] * 100
         # if there were females in that year
         if yr in females.index else 0
         # take each year
         for yr in year_total.index]

# add % line plot so subplots at [1,1]
fig.add_trace(
    go.Scatter(
        x=year_total.index,
        y=pct_f,
        mode="lines+markers",
        name="% Female",
        line=dict(color="#DD7D45", width=2),
        hovertemplate="Year %{x}: %{y:.1f}% female<extra></extra>"
    ),
    row=1, col=1
)

# 50% line
fig.add_hline(y=50, line=dict(color="grey", dash="dash", width=1), row=1, col=1)

# sex counts
# for each sex
for sex in ["M", "F"]:
    # take only the current sex count per year
    sub = part[part["Sex"] == sex].set_index("Year")
    # take count
    counts = [sub.loc[yr, "count"] if yr in sub.index else 0 for yr in year_total.index]

    # add subplot at [2,1]
    fig.add_trace(
        go.Scatter(
            x=year_total.index, 
            y=counts,
            mode="lines+markers",
            name=label_sex[sex],
            line=dict(color=color_sex[sex], width=2),
            marker=dict(size=4),
            hovertemplate=f"{label_sex[sex]} — Year %{{x}}: %{{y}}<extra></extra>",
        ),
        row=2, col=1
    )

# global chart layout config
fig.update_layout(
    title=dict(text="Participants by Year and Sex", font_size=22),
    template="plotly_white",
    height=620,
    legend=dict(x=1.01, y=0.5, xanchor="left"),
    margin=dict(l=60, r=120, t=80, b=50),
)

fig.update_xaxes(tickmode="linear", dtick=4, row=1, col=1)
fig.update_xaxes(title_text="Year", tickmode="linear", dtick=4, row=2, col=1)
fig.update_yaxes(title_text="Female [%]", ticksuffix="%", range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text="Athletes [#]", row=2, col=1)

fig.show()

In [29]:
avg_ages = df.groupby(["Year", "Season"])["Age"].mean().reset_index()
summer_ages = avg_ages[avg_ages["Season"] == "Summer"]
winter_ages = avg_ages[avg_ages["Season"] == "Winter"]

fig = make_subplots(
  rows=1,
  cols=1,
  shared_xaxes=True,
  shared_yaxes=True
)

fig.add_trace(
  go.Scatter(
      x=summer_ages['Year'],
      y=summer_ages['Age'],
      mode="lines+markers",
      name="Summer Games",
      line=dict(width=2, color="#E83E2E"),
      marker=dict(size=4),
  ),
  row=1, col=1
)

fig.add_trace(
  go.Scatter(
      x=winter_ages['Year'],
      y=winter_ages['Age'],
      mode="lines+markers",
      name="Winter Games",
      line=dict(width=2, color="#5052F9"),
      marker=dict(size=4),
  ),
  row=1, col=1
)

fig.update_layout(
    title=dict(text="Average Athletes' Age by Year", font_size=22),
    template="plotly_white",
    height=450,
    legend=dict(x=1.01, y=0.5, xanchor="left"),
    margin=dict(l=60, r=120, t=80, b=50),
)

min_age = avg_ages.min()
max_age = avg_ages.max()

fig.update_xaxes(title_text="Year", tickmode="linear", dtick=4)
fig.update_yaxes(title_text="Average Age", range=[min_age, max_age], dtick=1)

fig.show()

In [30]:
oldest_athlete = df[df["Age"] == df["Age"].max()]
oldest_athlete

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
257054,128719,John Quincy Adams Ward,M,97.0,NaN,NaN,United States,USA,1928 Summer,1928,Summer,Amsterdam,Art Competitions,"Art Competitions Mixed Sculpturing, Statues",NaN


In [31]:
yougest_athlete = df[df["Age"] == df["Age"].min()]
yougest_athlete

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
142882,71691,Dimitrios Loundras,M,10.0,NaN,NaN,Ethnikos Gymnastikos Syllogos,GRE,1896 Summer,1896,Summer,Athina,Gymnastics,"Gymnastics Men's Parallel Bars, Teams",Bronze


## Average Physical Profile by Sport
The scatter plot shows the average height and weight of athletes across 10 selected sports.
Each point represents the average physical profile of athletes in that discipline throughout Olympic history.
A clear pattern emerges: sports requiring explosive power and height (Basketball, Volleyball, Rowing) cluster in the top-right, while sports favoring lighter and shorter athletes (Gymnastics) sit in the bottom-left. Weightlifting is a notable outlier, athletes are short but very heavy.

In [ ]:
sport_selezionati = [
    'Basketball', 'Gymnastics', 'Weightlifting',
    'Swimming', 'Wrestling', 'Rowing',
    'Boxing', 'Cycling', 'Volleyball', 'Judo'
]

df_sport = (df[df['Sport'].isin(sport_selezionati)]
              .dropna(subset=['Height', 'Weight'])
              .groupby('Sport')[['Height', 'Weight']]
              .mean()
              .reset_index())

fig = px.scatter(
    df_sport,
    x='Height',
    y='Weight',
    text='Sport',
    title='Average Physical Profile by Sport',
    labels={'Height': 'Average Height (cm)', 'Weight': 'Average Weight (kg)'},
)

fig.update_traces(
    marker=dict(size=12, color='#6272a4'),
    textposition='top center'
)

fig.update_layout(template='plotly_white')

fig.show()

## Dominance by Sport and Country
The heatmap shows the medal share (%) for the top 15 countries and top 15 sports by total medals.
Darker cells indicate higher dominance. The USA stands out in Swimming and Athletics, while Russia (RUS) shows strength in Gymnastics and Cross Country Skiing. Italy (ITA) and France (FRA) dominate Fencing, Canada (CAN) leads in Ice Hockey, and the Netherlands (NED) in Speed Skating.
Note that the Soviet Union (URS) and East Germany (GDR) appear as separate entries, which dilutes their historical dominance across multiple country codes.

In [37]:
# medaglie per paese e sport
noc_sport = (medals.groupby(['NOC', 'Sport'])
                   .size()
                   .reset_index(name='noc_sport_count'))

# totale medaglie per sport
total_sport = (medals.groupby('Sport')
                     .size()
                     .reset_index(name='total_sport'))

# merge e calcolo share
noc_sport = noc_sport.merge(total_sport, on='Sport')
noc_sport['share'] = noc_sport['noc_sport_count'] / noc_sport['total_sport'] * 100

# top 15 sport per numero totale di medaglie
top_sports = (total_sport.nlargest(15, 'total_sport')['Sport'].tolist())

# top 15 paesi per medaglie totali
top_noc = (medals['NOC'].value_counts().head(15).index.tolist())

# filtra
heatmap_data = (noc_sport[noc_sport['Sport'].isin(top_sports) & 
                           noc_sport['NOC'].isin(top_noc)]
                .pivot(index='NOC', columns='Sport', values='share')
                .fillna(0))

fig = px.imshow(
    heatmap_data,
    title='Dominance by Sport and Country (medal share %)',
    labels=dict(color='Share (%)'),
    color_continuous_scale='Blues',
    aspect='auto'
)

fig.update_layout(
    template='plotly_white',
    height=600,
    xaxis_title='Sport',
    yaxis_title='Country'
)

fig.show()

In [61]:
# calcola share (riusa noc_sport calcolato prima)
dominatori = (noc_sport[noc_sport['Sport'].isin(top_sports)]
                .sort_values('share', ascending=False)
                .groupby('Sport')
                .first()
                .reset_index())[['Sport', 'NOC', 'share']]

dominatori['label'] = dominatori['NOC'] + ' (' + dominatori['share'].round(1).astype(str) + '%)'

fig = px.bar(
    dominatori.sort_values('share'),
    x='share',
    y='Sport',
    orientation='h',
    text='label',
    title='Who dominates each sport? (historical medal share %)',
    labels={'share': 'Share (%)', 'Sport': ''},
    color='share',
    color_continuous_scale='Blues',
)

fig.update_traces(textposition='outside')
fig.update_layout(
    template='plotly_white',
    height=600,
    showlegend=False,
    coloraxis_showscale=False,
)
fig.update_xaxes(range=[0, 42])
fig.show()

For each of the top 15 sports by total medals, this chart shows the country with the highest historical medal share.
The USA leads in Swimming (~32%) and Athletics (27.2%), meaning roughly 1 in 3 swimming medals has gone to an American athlete. Russia dominates Gymnastics and Cross Country Skiing, Germany leads in Canoeing and Rowing, and Japan is the top nation in Judo. The color intensity reflects the degree of dominance — darker bars indicate a stronger historical monopoly on that sport.